# Reliance Industries Stock Analysis & Forecasting

This notebook provides a clean, chronological workflow for Reliance Industries stock analysis and forecasting.

### Workflow
1. Data loading and validation
2. Data preprocessing and train/test split
3. Exploratory data analysis
4. SARIMA, Naive Baseline, and XGBoost
5. Additional forecasting models: ETS, Prophet, and State Space
6. Hold-out model comparison
7. Leakage audit and causality-safe feature engineering
8. LightGBM and CatBoost
9. Advanced technical indicators
10. Walk-forward validation
11. SHAP explainability
12. External market factors
13. Streamlit dashboard
14. MLflow and production artifact export

In [ ]:
#Load the dataset

import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/My Drive/DS/Time_series_forecasting/Company_stock_prices.csv")
print(df.head())

print(df.head())
print(df.shape)
print(df.columns)

In [ ]:
!pip install xgboost statsmodels

In [ ]:
pip install prophet

In [ ]:
#Convert Date

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print(df.head())
print(df.tail())

In [ ]:
#Verify the date range
print("Start Date:", df["Date"].min())
print("End Date:", df["Date"].max())

In [ ]:
#Check missing values
print(df.isnull().sum())

In [ ]:
#Check duplicate dates
print("Duplicate dates:", df["Date"].duplicated().sum())

In [ ]:
ohlc_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume"
]

df = df[ohlc_columns]

print(df.info())
print(df.describe())

# Data Preprocessing & Train-Test Splitting

In [ ]:
#Check data types
print(df.dtypes)

Check OHLC consistency

For every trading day:

High should be ≥ Open and Close

Low should be ≤ Open and Close

In [ ]:
print("High < Open:",
      (df["High"] < df["Open"]).sum())

print("High < Close:",
      (df["High"] < df["Close"]).sum())

print("Low > Open:",
      (df["Low"] > df["Open"]).sum())

print("Low > Close:",
      (df["Low"] > df["Close"]).sum())

In [ ]:
#Remove duplicates
df = df.drop_duplicates(subset=["Date"])

In [ ]:
#Create the training and test sets
train = df[df["Date"] < "2023-01-01"].copy()

test = df[df["Date"] >= "2023-01-01"].copy()

print("Training:")
print(train["Date"].min(), "to", train["Date"].max())
print("Rows:", len(train))

print("\nTesting:")
print(test["Date"].min(), "to", test["Date"].max())
print("Rows:", len(test))

In [ ]:
#Visualize the split
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(train["Date"], train["Close"], label="Training")
plt.plot(test["Date"], test["Close"], label="Testing")

plt.title("Reliance Industries — Training vs Testing Data")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)

plt.show()

#EDA & Trend Identification

Start with Closing Price

The first thing we want to understand is how Reliance's price moved over time.

| Analysis          | Feature                  |
| ----------------- | ------------------------ |
| Price trend       | `Close`                  |
| Daily movement    | `Daily_Return`           |
| Short-term trend  | `MA_20`                  |
| Long-term trend   | `MA_200`                 |
| Volatility        | `Volatility_20`          |
| Market activity   | `Volume`                 |
| Extreme movements | Largest absolute returns |

workflow

Closing Price─────→Daily Returns─────→20-Day MA  (Short-Term Trend)─────→200-Day MA(Long-Term Trend)─────→20-Day Volatility─────→Volume Analysis─────→Major Price Movements─────→External Event Investigation

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Close"])

plt.title("Reliance Industries Closing Price — 2020 to 2023")
plt.xlabel("Date")
plt.ylabel("Closing Price")

plt.grid(True)
plt.show()

Calculate Daily Returns

Price alone doesn't tell us how volatile the stock is.

In [ ]:
#Create daily percentage returns:
df["Daily_Return"] = df["Close"].pct_change() * 100

#View the result:
print(df[["Date", "Close", "Daily_Return"]].head(10))

#Plot
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Daily_Return"])

plt.title("Reliance Industries Daily Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return (%)")

plt.grid(True)
plt.show()

Short-Term Trend — Moving Average

We'll use a 20-day moving average to represent approximately one trading month.

When:

Close > MA20

→ short-term price is generally stronger.

When:

Close < MA20

→ short-term price is generally weaker.

In [ ]:
df["MA_20"] = df["Close"].rolling(window=20).mean()

#Plot
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Close"], label="Close")
plt.plot(df["Date"], df["MA_20"], label="20-Day Moving Average")

plt.title("Reliance Industries — Short-Term Trend")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)

plt.show()

Long-Term Trend — 200-Day Moving Average

Now create a 200-day moving average.

Why 200 days?

A stock typically has around 250 trading sessions per year.

Therefore, a 200-day moving average gives us a useful approximation of the long-term market trend.

In [ ]:
#create a 200-day moving average
df["MA_200"] = df["Close"].rolling(window=200).mean()

#Plot
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Close"], label="Close")
plt.plot(df["Date"], df["MA_200"], label="200-Day Moving Average")

plt.title("Reliance Industries — Long-Term Trend")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#Short-Term + Long-Term Together

plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Close"], label="Close")
plt.plot(df["Date"], df["MA_20"], label="20-Day MA")
plt.plot(df["Date"], df["MA_200"], label="200-Day MA")

plt.title("Reliance Industries — Short & Long-Term Trends")
plt.xlabel("Date")
plt.ylabel("Price")

plt.legend()
plt.grid(True)

plt.show()

Volatility Analysis

Interpretation

Higher volatility:

→ larger price fluctuations
→ greater uncertainty

Lower volatility:

→ relatively stable price movement.

In [ ]:
#Now calculate rolling volatility.
df["Volatility_20"] = df["Daily_Return"].rolling(window=20).std()

#Plot:
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Volatility_20"])

plt.title("Reliance Industries — 20-Day Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")

plt.grid(True)
plt.show()

Trading Volume

Trading volume in stocks refers to the total number of shares or contracts of a particular stock that are bought and sold during a specific period, such as a day, week, or month.

our dataset contains Volume, so we should examine it.

We can later compare large volume spikes with large price movements.

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Volume"])

plt.title("Reliance Industries Trading Volume")
plt.xlabel("Date")
plt.ylabel("Volume")

plt.grid(True)
plt.show()

#Identify Major Price-Movement Days

This is important because these dates become candidates for our external-event investigation.

In [ ]:
#find the days where the stock moved the most
largest_moves = df.loc[
    df["Daily_Return"].abs().nlargest(10).index,
    ["Date", "Open", "High", "Low", "Close", "Daily_Return", "Volume"]
]

print(largest_moves.sort_values("Date"))

#External Events

This is important because these are the dates we should investigate, rather than choosing events arbitrarily.

| Period | External factor                    | Potential effect on RIL                      |
| ------ | ---------------------------------- | -------------------------------------------- |
| 2020   | COVID-19 / lockdown                | Negative market & oil shock                  |
| 2020   | Jio investments                    | Positive sentiment / business transformation |
| 2022   | Russia-Ukraine / high crude prices | Major energy-market impact                   |
| 2022   | Windfall tax introduction          | Negative for oil/refining sentiment          |
| 2022   | Windfall tax reduction             | Positive reaction                            |
| 2023   | Jio Financial Services demerger    | Corporate restructuring / price adjustment   |

In [ ]:
#find the biggest movements

largest_moves = df.loc[
    df["Daily_Return"].abs().nlargest(15).index,
    ["Date", "Open", "High", "Low", "Close", "Daily_Return", "Volume"]
]

largest_moves = largest_moves.sort_values("Date")

print(largest_moves.to_string(index=False))

largest_moves = df.loc[
    df["Daily_Return"].abs().nlargest(15).index,
    ["Date", "Open", "High", "Low", "Close", "Daily_Return", "Volume"]
]

print(largest_moves.sort_values("Date").to_string(index=False))

#Validate suspicious price movements

This will show approximately 5 calendar days before and after each event.

In [ ]:
#Check prices around each suspicious date
suspicious_dates = [
    "2022-01-21",
    "2022-04-20",
    "2023-07-20"
]

for date in suspicious_dates:
    print(f"\n{'='*60}")
    print(f"Data around {date}")
    print(f"{'='*60}")

    target_date = pd.to_datetime(date)

    result = df[
        (df["Date"] >= target_date - pd.Timedelta(days=5)) &
        (df["Date"] <= target_date + pd.Timedelta(days=5))
    ][["Date", "Open", "High", "Low", "Close", "Daily_Return"]]

    print(result.to_string(index=False))

In [ ]:
#Check for extreme daily returns

extreme_moves = df[
    df["Daily_Return"].abs() > 10
][["Date", "Close", "Daily_Return", "Volume"]]

print(extreme_moves.to_string(index=False))

#Calculate the Close-to-Close price ratio

df["Close_Ratio"] = df["Close"] / df["Close"].shift(1)

print(
    df.loc[
        df["Daily_Return"].abs() > 10,
        ["Date", "Close", "Close_Ratio", "Daily_Return"]
    ].to_string(index=False)
)

#Visualize the suspicious periods


import matplotlib.pyplot as plt

date = pd.to_datetime("2022-04-20")

window = df[
    (df["Date"] >= date - pd.Timedelta(days=30)) &
    (df["Date"] <= date + pd.Timedelta(days=30))
]

plt.figure(figsize=(12, 5))

plt.plot(window["Date"], window["Close"], marker="o")

plt.axvline(date, linestyle="--", label="Suspicious Date")

plt.title("Reliance Stock Price Around 20-Apr-2022")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.grid(True)

plt.show()

date = pd.to_datetime("2022-01-21")
date = pd.to_datetime("2023-07-20")
date = pd.to_datetime("2023-04-20")

## Evaluation metrics

We'll use RMSE, MAE, and MAPE throughout so every model (including the naive baseline and XGBoost below) is scored the same way.

In [ ]:
#preserve the original data and create a validation flag.

df["Extreme_Movement"] = df["Daily_Return"].abs() > 10

#auditable record of unusual observations.

print(
    df[df["Extreme_Movement"]][
        ["Date", "Close", "Daily_Return", "Volume"]
    ].to_string(index=False)
)

#compare the raw OHLC sequence immediately before and after the dates.

for date in ["2022-01-21", "2022-04-20"]:

    target = pd.to_datetime(date)

    print("\n" + "=" * 70)
    print(f"PRICE DATA AROUND {date}")
    print("=" * 70)

    print(
        df[
            (df["Date"] >= target - pd.Timedelta(days=3)) &
            (df["Date"] <= target + pd.Timedelta(days=3))
        ][
            ["Date", "Open", "High", "Low", "Close", "Volume"]
        ].to_string(index=False)
    )

Why are we checking this?

We want to see whether:

Close

and

Adj Close

behave differently around these discontinuities.

In [ ]:
df[
    df["Date"].isin(
        pd.to_datetime([
            "2022-01-20",
            "2022-01-21",
            "2022-04-19",
            "2022-04-20",
            "2022-07-19",
            "2022-07-20"
        ])
    )
][
    ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
].to_string(index=False)

#Important observations

|20-Jan-2021: +16.85%|
|21-Jan-2022: −21.79%|
|20-Apr-2022: −35.12%|
|19-Oct-2022: +13.09%|
|20-Jul-2023: −8.41%|

#Model Building & Evaluation

Model 1 — SARIMA

Captures:

Time-series patterns and temporal dependencies

Model 2 — XGBoost

Captures:

Non-linear relationships using engineered lag and rolling features

Prepare the data for SARIMA.

4.1 Select the target

For this project, we'll forecast Reliance Industries' Closing Price.

In [ ]:
import numpy as np
# Target variable
train_close = train.set_index("Date")["Close"]
test_close = test.set_index("Date")["Close"]

# Assign names to the series for better clarity, especially for statsmodels
train_close.name = 'Close'
test_close.name = 'Close'

print("Training observations:", len(train_close))
print("Testing observations:", len(test_close))
display(train_close.head())
display(test_close.head())

#Check stationarity

SARIMA works with time-series data, and we need to determine whether the series is stationary.

We'll use the Augmented Dickey-Fuller (ADF) test.

p-value < 0.05 --->Reject H₀---->Series is stationary

&

p-value >= 0.05---->Cannot reject H₀----Series is non-stationary

If our p-value is ≥ 0.05, that's not a problem—we'll use differencing as part of SARIMA.

if p-value < 0.05
then the first difference is stationary, and we can use: d = 1, in SARIMA.

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(train_close.dropna())

print("ADF Statistic:", result[0])
print("p-value:", result[1])


In [ ]:
#First difference
train_diff = train_close.diff().dropna()

result_diff = adfuller(train_diff)

print("ADF Statistic:", result_diff[0])
print("p-value:", result_diff[1])

#Visualize the differenced series
#You should see a series fluctuating around a relatively stable mean rather than continuously trending upward/downward

import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(train_diff)

plt.title("Differenced Reliance Industries Closing Price")
plt.xlabel("Date")
plt.ylabel("Price Difference")

plt.grid(True)
plt.show()

#What SARIMA means

SARIMA(p,d,q)(P,D,Q,s)

for example

SARIMA(1,1,1)

p = 1 → autoregressive component
d = 1 → first-order differencing
q = 1 → moving-average component

In [ ]:
print("Training observations:", len(train_close))
print("Testing observations:", len(test_close))

result = adfuller(train_close.dropna())
print("\nOriginal Series")
print("ADF Statistic:", result[0])
print("p-value:", result[1])

train_diff = train_close.diff().dropna()

result_diff = adfuller(train_diff)
print("\nFirst Difference")
print("ADF Statistic:", result_diff[0])
print("p-value:", result_diff[1])

#Determine Differencing (d)
#the original closing price:
p-value = 0.7866

#Since:

0.7866 > 0.05
#we cannot reject the null hypothesis of a unit root.

#We need one order of differencing.
# So our SARIMA parameter is:

d = 1


Determine p and q

Now we need to determine the AR (p) and MA (q) components.

We'll use:

ACF → helps identify q
PACF → helps identify p

Importantly, we'll calculate these using the stationary first-differenced training series, not the original non-stationary prices.

ACF--->Possible MA order--->q

PACF----->Possible AR order--->p

For example, if the PACF cuts off strongly around lag 1 and the ACF also cuts off around lag 1, we might investigate:

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))

plot_acf(
    train_diff,
    lags=40,
    ax=ax
)

plt.title("ACF - Differenced Reliance Closing Price")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.grid(True)

plt.show()

In [ ]:
#PACF
fig, ax = plt.subplots(figsize=(14, 6))

plot_pacf(
    train_diff,
    lags=40,
    ax=ax,
    method="ywm"
)

plt.title("PACF - Differenced Reliance Closing Price")
plt.xlabel("Lag")
plt.ylabel("Partial Autocorrelation")
plt.grid(True)

plt.show()

## Selecting SARIMA order (p, q)

The ACF/PACF plots above give a rough sense of which lags matter, but reading them by eye is subjective. We'll confirm the exact order with a small AIC-based grid search over p, q ∈ {0, 1, 2}, keeping d = 1 (from the ADF test above).

No seasonal terms are included — Reliance's closing price is a trending series with no clear repeating cycle, so a seasonal component isn't justified here and would only add unnecessary complexity.

In [ ]:
import warnings
import numpy as np
warnings.filterwarnings("ignore")

from statsmodels.tsa.statespace.sarimax import SARIMAX

best_aic = float("inf")
best_order = None

# Expand the search range slightly and remove 'enforce' parameters to use defaults
# which are often more stable.
for p in range(0, 4):
    for q in range(0, 4):
        try:
            candidate = SARIMAX(
                train_close,
                order=(p, 1, q),
            ).fit(disp=False)

            # Ensure AIC is a valid number before comparison
            if not np.isnan(candidate.aic) and candidate.aic < best_aic:
                best_aic = candidate.aic
                best_order = (p, 1, q)
        except Exception as e:
            # print(f"SARIMAX({p},1,{q}) failed: {e}") # For debugging specific errors
            continue

# Fallback in case no valid order was found after the grid search
if best_order is None:
    print("Warning: No optimal SARIMA order found. Defaulting to (1, 1, 1).")
    best_order = (1, 1, 1) # A common and robust default order

print("Best order (p, d, q):", best_order)
print("Best AIC:", best_aic)

## Fit the final SARIMA model

Refit on the selected order and forecast across the full test period (198 trading days).

In [ ]:
sarima_model = SARIMAX(
    train_close,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(sarima_model.summary())

sarima_forecast_result = sarima_model.get_forecast(steps=len(test_close))
sarima_forecast = sarima_forecast_result.predicted_mean
sarima_conf_int = sarima_forecast_result.conf_int(alpha=0.05)

sarima_forecast.index = test_close.index
sarima_conf_int.index = test_close.index

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(train_close.index, train_close, label="Training")
plt.plot(test_close.index, test_close, label="Actual (Test)")
plt.plot(sarima_forecast.index, sarima_forecast, label="SARIMA Forecast")
plt.fill_between(
    sarima_conf_int.index,
    sarima_conf_int.iloc[:, 0],
    sarima_conf_int.iloc[:, 1],
    color="gray",
    alpha=0.2,
    label="95% Confidence Interval"
)

plt.title("SARIMA — Actual vs Forecast")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_forecast(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"{model_name} Evaluation")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"MAPE: {mape:.2f}%")

    return {"Model": model_name, "RMSE": rmse, "MAE": mae, "MAPE": mape}

sarima_metrics = evaluate_forecast(test_close.values, sarima_forecast.values, "SARIMA")

## Naive baseline (reference point)

Before trusting SARIMA's error numbers, compare them against the simplest possible forecast: assume every future day equals the last observed training price. Any model that can't beat this isn't adding value.

In [ ]:
naive_forecast = pd.Series(
    [train_close.iloc[-1]] * len(test_close),
    index=test_close.index
)

naive_metrics = evaluate_forecast(test_close.values, naive_forecast.values, "Naive Baseline")

## XGBoost model

XGBoost can't use raw dates or work autoregressively like SARIMA, so we engineer lag and rolling features that give it the same kind of information explicitly, as tabular features.

In [ ]:
feature_df = df[["Date", "Close", "Volume"]].copy()

for lag in [1, 2, 3, 5, 10]:
    feature_df[f"Lag_{lag}"] = feature_df["Close"].shift(lag)

# shift(1) before rolling so today's row never sees today's own close
feature_df["Rolling_Mean_5"] = feature_df["Close"].shift(1).rolling(window=5).mean()
feature_df["Rolling_Std_5"] = feature_df["Close"].shift(1).rolling(window=5).std()
feature_df["Rolling_Mean_20"] = feature_df["Close"].shift(1).rolling(window=20).mean()

feature_df["DayOfWeek"] = feature_df["Date"].dt.dayofweek
feature_df["Month"] = feature_df["Date"].dt.month

feature_df = feature_df.dropna().reset_index(drop=True)

print(feature_df.shape)
print(feature_df.head())

In [ ]:
# Same chronological cutoff as the SARIMA split
feature_train = feature_df[feature_df["Date"] < "2023-01-01"]
feature_test = feature_df[feature_df["Date"] >= "2023-01-01"]

feature_cols = [c for c in feature_df.columns if c not in ["Date", "Close"]]

X_train = feature_train[feature_cols]
y_train = feature_train["Close"]

X_test = feature_test[feature_cols]
y_test = feature_test["Close"]

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_forecast = xgb_model.predict(X_test)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(feature_train["Date"], y_train, label="Training")
plt.plot(feature_test["Date"], y_test, label="Actual (Test)")
plt.plot(feature_test["Date"], xgb_forecast, label="XGBoost Forecast")

plt.title("XGBoost — Actual vs Forecast")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
xgb_metrics = evaluate_forecast(y_test.values, xgb_forecast, "XGBoost")

### Feature importance

Which engineered features XGBoost actually relied on.

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.plot(kind="bar")
plt.title("XGBoost Feature Importance")
plt.ylabel("Importance")
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()

print(importances)

## Additional forecasting models

Testing alternative exponential-smoothing, Prophet, and state-space approaches using the same chronological 2023 hold-out test set. No test-set observations are used during fitting.

In [ ]:
# Additional model imports
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.structural import UnobservedComponents

warnings.filterwarnings("ignore")

# Reuse the same train/test Close series already defined above.


### Model 3 — Exponential Smoothing (ETS / Holt-Winters)

The series does not show a justified repeating seasonal cycle, so no seasonal component is used. Both additive and multiplicative trend specifications are tested; the lower training AIC is used to select the specification before the untouched 2023 test evaluation.

In [ ]:
ets_candidates = {}

for trend_name, trend in [("Additive Trend", "add"), ("Multiplicative Trend", "mul")]:
    ets_model = ExponentialSmoothing(
        train_close,
        trend=trend,
        seasonal=None,
        initialization_method="estimated"
    ).fit()
    ets_candidates[trend_name] = ets_model
    print(f"ETS {trend_name}: AIC = {ets_model.aic:.2f}")

best_ets_name = min(ets_candidates, key=lambda name: ets_candidates[name].aic)
ets_model = ets_candidates[best_ets_name]
ets_forecast = ets_model.forecast(len(test_close))
ets_forecast.index = test_close.index

print("Selected ETS specification:", best_ets_name)

ets_metrics = evaluate_forecast(
    test_close.values,
    ets_forecast.values,
    "ETS"
)


In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train_close.index, train_close, label="Training")
plt.plot(test_close.index, test_close, label="Actual (Test)")
plt.plot(test_close.index, ets_forecast, label=f"ETS Forecast ({best_ets_name})")
plt.title("ETS — Actual vs Forecast")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.show()


### Model 4 — Prophet

Prophet is tested twice: once with its default seasonality configuration and once with all built-in seasonalities disabled. Because this is a trending, largely non-seasonal stock series, the seasonality-disabled specification is included as the simpler benchmark. The final test metrics are computed only after fitting each candidate on the training window.

**Dependency:** install `prophet` before running these cells.

In [ ]:
try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError(
        "Prophet is required for this section. Install it with `pip install prophet` "
        "and rerun the Prophet cells."
    ) from exc

prophet_train = train_close.reset_index()
prophet_train.columns = ["ds", "y"]

prophet_default = Prophet()
prophet_default.fit(prophet_train)

prophet_no_seasonality = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False
)
prophet_no_seasonality.fit(prophet_train)

# Evaluate on the exact held-out trading dates; do not create synthetic
# business-day observations that include exchange holidays.
prophet_test_future = pd.DataFrame({"ds": test_close.index})
prophet_default_forecast = prophet_default.predict(prophet_test_future)
prophet_no_seasonality_forecast = prophet_no_seasonality.predict(prophet_test_future)

prophet_default_pred = prophet_default_forecast["yhat"].to_numpy()
prophet_no_seasonality_pred = prophet_no_seasonality_forecast["yhat"].to_numpy()

prophet_default_metrics = evaluate_forecast(
    test_close.values, prophet_default_pred, "Prophet (Default Seasonality)"
)
prophet_no_seasonality_metrics = evaluate_forecast(
    test_close.values, prophet_no_seasonality_pred, "Prophet (No Seasonality)"
)

# The following plotting code is moved to the relevant State Space model cell.
# plt.figure(figsize=(14, 6))
# plt.plot(train_close.index, train_close, label="Training")
# plt.plot(test_close.index, test_close, label="Actual (Test)")
# plt.plot(test_close.index, state_space_forecast, label=f"State Space Forecast ({best_state_space_name})")
# plt.fill_between(
#     test_close.index,
#     state_space_conf_int.iloc[:, 0].values,
#     state_space_conf_int.iloc[:, 1].values,
#     alpha=0.15,
#     label="State Space 95% CI"
# )
# plt.title("State Space Model — Actual vs Forecast")
# plt.xlabel("Date")
# plt.ylabel("Closing Price")
# plt.legend()
# plt.grid(True)
# plt.show()

The two Prophet specifications are both retained in the comparison table because seasonality configuration is a model-design choice rather than something that should be selected using the held-out 2023 test set. This keeps the final test set strictly for evaluation.

### Model 5 — State Space Model

A structural state-space model is used as a probabilistic alternative to SARIMA. Both a local-level and local-linear-trend specification are fitted; the lower training AIC is selected before evaluating on the held-out test set.

In [ ]:
state_space_candidates = {}

state_space_candidates["Local Level"] = UnobservedComponents(
    train_close,
    level="local level"
).fit(disp=False)

state_space_candidates["Local Linear Trend"] = UnobservedComponents(
    train_close,
    level="local linear trend"
).fit(disp=False)

for name, model in state_space_candidates.items():
    print(f"State Space {name}: AIC = {model.aic:.2f}")

best_state_space_name = min(
    state_space_candidates,
    key=lambda name: state_space_candidates[name].aic
)
state_space_model = state_space_candidates[best_state_space_name]
state_space_forecast_result = state_space_model.get_forecast(steps=len(test_close))
state_space_forecast = state_space_forecast_result.predicted_mean
state_space_conf_int = state_space_forecast_result.conf_int(alpha=0.05)
state_space_forecast.index = test_close.index
state_space_conf_int.index = test_close.index

print("Selected state-space specification:", best_state_space_name)

state_space_metrics = evaluate_forecast(
    test_close.values,
    state_space_forecast.values,
    "State Space"
)


In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train_close.index, train_close, label="Training")
plt.plot(test_close.index, test_close, label="Actual (Test)")
plt.plot(test_close.index, state_space_forecast, label=f"State Space Forecast ({best_state_space_name})")
plt.fill_between(
    test_close.index,
    state_space_conf_int.iloc[:, 0].values,
    state_space_conf_int.iloc[:, 1].values,
    alpha=0.15,
    label="State Space 95% CI"
)
plt.title("State Space Model — Actual vs Forecast")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.show()


## model comparison

The original Naive Baseline, SARIMA, and XGBoost rows are preserved unchanged. The additional models are appended using the same RMSE, MAE, and MAPE function and the same 2023 hold-out window.

In [ ]:
comparison_df = pd.DataFrame([
    naive_metrics,
    sarima_metrics,
    xgb_metrics,
    ets_metrics,
    prophet_default_metrics,
    prophet_no_seasonality_metrics,
    state_space_metrics
]).set_index("Model")

print(comparison_df)

comparison_df[["RMSE", "MAE"]].plot(kind="bar", figsize=(10, 6))
plt.title("Model Comparison — RMSE & MAE")
plt.ylabel("Error")
plt.xticks(rotation=20)
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()

comparison_df.to_csv("model_comparison.csv")

## Hold-out Model Comparison — Interpretation

The comparison table above provides a common evaluation framework for the Naive Baseline, SARIMA, XGBoost, ETS, Prophet, and State Space models.


### Reading the comparison table


#Key Performance Takeaways

Error Reduction:
XGBoost achieved an RMSE of ~16.5, compared to ~92.8 for the baseline and SARIMA models. This indicates that tree-based gradient boosting captured non-linear relationships and feature interactions in the time series far better than traditional linear/statistical modeling.

MAPE (Mean Absolute Percentage Error):
The MAPE dropped from ~19.99% down to 3.53%, meaning your model's predictions are, on average, within about 3.5% of the actual stock price during the test period.

Why XGBoost Won:
Standard time-series models like SARIMA often struggle with complex financial data unless stationarity is perfectly tuned. XGBoost thrives when provided with rich feature engineering (such as lagged returns, rolling averages, and technical indicators).


Lower RMSE/MAE/MAPE is better. Both SARIMA and XGBoost should beat the naive baseline — if either doesn't, that model isn't adding value over "assume no change" and is worth revisiting rather than reporting as-is.

XGBoost has access to Volume and calendar features that SARIMA doesn't, so it may pick up short-term wiggles SARIMA smooths over; SARIMA, in turn, tends to give a more stable trend-following forecast with honest uncertainty bands (the shaded region above) that XGBoost's point predictions don't provide out of the box. Which one is "better" here depends on the actual numbers this run produces on your data — worth looking at the table above rather than assuming an outcome ahead of time.

# Advanced Forecasting Pipeline

The following sections extend the original analysis with stricter validation, leakage-safe features, additional gradient-boosting models, explainability, external market drivers, dashboard integration, and experiment tracking.


# Advanced Forecasting Pipeline

The following sections extend the original analysis with stricter validation, leakage-safe features, additional gradient-boosting models, explainability, external market drivers, dashboard integration, and experiment tracking.

## Section 2: Leakage Audit & Fix

### Subtask:
Audit and fix the feature engineering pipeline to ensure strict causality and prevent look-ahead bias.

In [ ]:
import pandas as pd

# Re-engineering features with explicit shift(1) for rolling stats to ensure no leakage
# Lags naturally look back, but rolling windows at index i must not include close[i]

feature_df_clean = df[['Date', 'Close', 'Volume']].copy()

# 1. Standard Lags
for lag in [1, 2, 3, 5, 10]:
    feature_df_clean[f'Lag_{lag}'] = feature_df_clean['Close'].shift(lag)

# 2. Rolling Statistics (explicitly shifted by 1 to represent 'previous day' info)
# This ensures that at time 't', we only know stats up to 't-1'
feature_df_clean['Rolling_Mean_5'] = feature_df_clean['Close'].shift(1).rolling(window=5).mean()
feature_df_clean['Rolling_Std_5'] = feature_df_clean['Close'].shift(1).rolling(window=5).std()
feature_df_clean['Rolling_Mean_20'] = feature_df_clean['Close'].shift(1).rolling(window=20).mean()

# 3. Calendar Features
feature_df_clean['DayOfWeek'] = feature_df_clean['Date'].dt.dayofweek
feature_df_clean['Month'] = feature_df_clean['Date'].dt.month

# Drop rows with NaNs resulting from lags/rolling windows
feature_df_clean = feature_df_clean.dropna().reset_index(drop=True)

print('Leakage Audit: Features regenerated with strict t-1 causality.')
print(f'New feature_df shape: {feature_df_clean.shape}')
display(feature_df_clean.head())

## Section 3: LightGBM Implementation

### Subtask:
Implement and evaluate a LightGBM regressor on the audited feature set.

### LightGBM Model Preparation

This section introduces a **LightGBM gradient-boosting regression model** using the causality-audited feature dataset.

The workflow:
1. Use `feature_df_clean`, which contains the corrected leakage-safe features.
2. Preserve the chronological train/test split used throughout the notebook.
3. Separate predictor variables from the target closing price.
4. Train LightGBM only on historical training observations.
5. Generate predictions for the unseen test period.
6. Evaluate the predictions using the same RMSE, MAE, and MAPE metrics used for the other forecasting models.

Using the audited feature set ensures that LightGBM is evaluated fairly without future information leaking into the training process.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np

# Split data using the chronological cutoff
feature_train = feature_df_clean[feature_df_clean['Date'] < '2023-01-01']
feature_test = feature_df_clean[feature_df_clean['Date'] >= '2023-01-01']

X_train = feature_train[feature_cols]
y_train = feature_train['Close']
X_test = feature_test[feature_cols]
y_test = feature_test['Close']

# Define and train LightGBM
lgbm_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    importance_type='gain'
)

lgbm_model.fit(X_train, y_train)

# Predict
lgbm_forecast = lgbm_model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, lgbm_forecast))
mae = mean_absolute_error(y_test, lgbm_forecast)
mape = np.mean(np.abs((y_test - lgbm_forecast) / y_test)) * 100

lgbm_metrics = {'Model': 'LightGBM', 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

print(f'LightGBM Trained.\nRMSE: {rmse:.2f}, MAE: {mae:.2f}, MAPE: {mape:.2f}%')


### LightGBM Evaluation & Feature Importance

This section evaluates the LightGBM model visually and examines which engineered variables contributed most strongly to its predictions.

Two outputs are produced:
- **Actual vs. Predicted chart:** compares the LightGBM forecast with the real closing prices throughout the hold-out test period.
- **Feature-importance chart:** ranks the input variables according to their contribution to the trained LightGBM model.

Together, these visualizations help assess both **forecast accuracy** and **model interpretability**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Plot Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(feature_test['Date'], y_test, label='Actual (Test)', color='black', alpha=0.7)
plt.plot(feature_test['Date'], lgbm_forecast, label='LightGBM Forecast', color='orange', linestyle='--')
plt.title('LightGBM Model: Actual vs Forecasted Stock Price')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.legend()
plt.grid(True)
plt.show()

# Plot Feature Importance
importances = pd.Series(lgbm_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
importances.plot(kind='bar', color='skyblue')
plt.title('LightGBM Feature Importance (Gain)')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

print('LightGBM Visualization and Feature Importance analysis completed.')

## Section 4: CatBoost Implementation

### Subtask:
Implement and evaluate a CatBoost regressor on the audited feature set.

### CatBoost Model Training & Evaluation

This section implements a **CatBoost regression model** using the leakage-audited feature set.

The model is trained chronologically:
- training observations contain only historical information;
- the test set represents the unseen forecasting period; and
- no future test observations are used during training; and
- performance is measured using RMSE, MAE, and MAPE.

The purpose is to determine whether CatBoost can provide competitive forecasting performance while maintaining the same causality and evaluation standards applied to the other machine-learning models.

In [ ]:
try:
    from catboost import CatBoostRegressor
except ImportError:
    !pip install catboost
    from catboost import CatBoostRegressor

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Prepare data splits
feature_train = feature_df_clean[feature_df_clean['Date'] < '2023-01-01']
feature_test = feature_df_clean[feature_df_clean['Date'] >= '2023-01-01']

X_train = feature_train[feature_cols]
y_train = feature_train['Close']
X_test = feature_test[feature_cols]
y_test = feature_test['Close']

# Initialize and train CatBoost
cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_seed=42,
    verbose=False
)

cat_model.fit(X_train, y_train)

# Forecast
cat_forecast = cat_model.predict(X_test)

# Calculate Metrics
rmse = np.sqrt(mean_squared_error(y_test, cat_forecast))
mae = mean_absolute_error(y_test, cat_forecast)
mape = np.mean(np.abs((y_test - cat_forecast) / y_test)) * 100

cat_metrics = {'Model': 'CatBoost', 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

print(f'CatBoost Trained.\nRMSE: {rmse:.2f}, MAE: {mae:.2f}, MAPE: {mape:.2f}%')

### CatBoost Forecast Visualization & Interpretation

This section provides two complementary views of the CatBoost model.

First, the predicted closing prices are plotted against the actual test-set prices to show how closely the model follows the observed market movement.

Second, CatBoost feature importance is visualized to identify which engineered variables have the greatest influence on the model's forecasts.

These outputs provide a practical combination of **prediction assessment** and **model interpretability**.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Visualize Actual vs CatBoost Forecast
plt.figure(figsize=(14, 6))
plt.plot(feature_test['Date'], y_test, label='Actual (Test)', color='black', alpha=0.7)
plt.plot(feature_test['Date'], cat_forecast, label='CatBoost Forecast', color='green', linestyle='--')
plt.title('CatBoost Model: Actual vs Forecasted Reliance Stock Price')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.legend()
plt.grid(True)
plt.show()

# 2. Extract and Plot Feature Importance
cat_importances = pd.Series(cat_model.get_feature_importance(), index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
cat_importances.plot(kind='bar', color='seagreen')
plt.title('CatBoost Feature Importance')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('CatBoost Visualization and Feature Importance Analysis completed.')

## Section 5: Advanced Technical Indicators

### Subtask:
Develop a suite of technical and market indicators including RSI, MACD, Bollinger Bands, and EMAs, ensuring strict causality by shifting the values.

### Core Technical Indicators

This section expands the forecasting feature set with commonly used technical-analysis indicators calculated from historical price information.

The initial indicators are:
- **EMA (Exponential Moving Average):** captures short- and medium-term price trends by giving greater weight to recent observations.
- **RSI (Relative Strength Index):** measures the strength of recent upward and downward price movements.
- **MACD (Moving Average Convergence Divergence):** captures changes in trend and momentum using the relationship between exponential moving averages.
- **Bollinger Bands:** describe the position of price relative to a rolling mean and volatility-based upper and lower bands.

The calculations are designed to remain causality-safe so that the model does not use information that would not have been available at prediction time.

In [ ]:
import pandas as pd
import numpy as np

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# 1. Create a fresh copy for technical features
df_tech = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

# 2. EMAs
df_tech['EMA_20'] = df_tech['Close'].ewm(span=20, adjust=False).mean()
df_tech['EMA_50'] = df_tech['Close'].ewm(span=50, adjust=False).mean()
df_tech['EMA_200'] = df_tech['Close'].ewm(span=200, adjust=False).mean()

# 3. RSI
df_tech['RSI'] = calculate_rsi(df_tech['Close'])

# 4. MACD
exp1 = df_tech['Close'].ewm(span=12, adjust=False).mean()
exp2 = df_tech['Close'].ewm(span=26, adjust=False).mean()
df_tech['MACD'] = exp1 - exp2
df_tech['MACD_Signal'] = df_tech['MACD'].ewm(span=9, adjust=False).mean()

# 5. Bollinger Bands (20-day)
sma_20 = df_tech['Close'].rolling(window=20).mean()
std_20 = df_tech['Close'].rolling(window=20).std()
df_tech['BB_Upper'] = sma_20 + (std_20 * 2)
df_tech['BB_Lower'] = sma_20 - (std_20 * 2)

# Shift all indicators to prevent leakage (today's feature = yesterday's indicator)
tech_cols = ['EMA_20', 'EMA_50', 'EMA_200', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower']
for col in tech_cols:
    df_tech[col] = df_tech[col].shift(1)

print('Advanced Technical Indicators calculated and shifted.')
print(df_tech[tech_cols].tail())


**comment**:
I will calculate ATR, ROC, and ADX, ensuring each is shifted by one day to maintain strict causality, then merge these with the existing technical features and drop the resulting NaNs to prepare the final feature set.

In [ ]:
def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(window=period).mean()

# 6. ATR (Volatility)
df_tech['ATR'] = calculate_atr(df_tech)

# 7. ROC (Rate of Change)
df_tech['ROC'] = df_tech['Close'].pct_change(periods=10) * 100

# 8. Simplified ADX logic (Directional Strength)
def calculate_adx(df, period=14):
    plus_dm = df['High'].diff()
    minus_dm = df['Low'].diff()
    plus_dm[plus_dm < 0] = 0
    minus_dm[minus_dm > 0] = 0
    tr = calculate_atr(df, period)
    plus_di = 100 * (plus_dm.rolling(window=period).mean() / tr)
    minus_di = 100 * (minus_dm.abs().rolling(window=period).mean() / tr)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    return dx.rolling(window=period).mean()

df_tech['ADX'] = calculate_adx(df_tech)

# Shift the new indicators
for col in ['ATR', 'ROC', 'ADX']:
    df_tech[col] = df_tech[col].shift(1)

# Drop NaNs created by rolling windows
df_tech_final = df_tech.dropna().reset_index(drop=True)

print('Advanced indicators (ATR, ROC, ADX) added and shifted.')
print(f'Technical Dataframe Shape: {df_tech_final.shape}')
display(df_tech_final[['Date', 'RSI', 'MACD', 'ATR', 'ROC', 'ADX']].head())

**comment**:
I will train the CatBoost model on the expanded technical feature set (17 features) and evaluate its performance against the previous baseline to quantify the improvement from advanced indicators.

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# 1. Define final feature columns excluding target and metadata
all_tech_cols = [c for c in df_tech_final.columns if c not in ['Date', 'Close', 'Open', 'High', 'Low', 'Adj Close']]

# 2. Chronological Split
tech_train = df_tech_final[df_tech_final['Date'] < '2023-01-01']
tech_test = df_tech_final[df_tech_final['Date'] >= '2023-01-01']

X_train_tech = tech_train[all_tech_cols]
y_train_tech = tech_train['Close']
X_test_tech = tech_test[all_tech_cols]
y_test_tech = tech_test['Close']

# 3. Train CatBoost with Advanced Indicators
cat_tech_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=False
)

cat_tech_model.fit(X_train_tech, y_train_tech)

# 4. Predict and Evaluate
cat_tech_pred = cat_tech_model.predict(X_test_tech)

rmse = np.sqrt(mean_squared_error(y_test_tech, cat_tech_pred))
mae = mean_absolute_error(y_test_tech, cat_tech_pred)
mape = np.mean(np.abs((y_test_tech - cat_tech_pred) / y_test_tech)) * 100

print(f'CatBoost with Advanced Technical Indicators:')
print(f'Features used: {len(all_tech_cols)}')
print(f'RMSE: {rmse:.2f}')
print(f'MAE:  {mae:.2f}')
print(f'MAPE: {mape:.2f}%')

# Compare with previous best (15.77)
if rmse < 15.77:
    print('\nPerformance IMPROVED with advanced indicators.')
else:
    print('\nPerformance did not improve over simple lags.')

## Section 1: Walk-Forward Validation

### Subtask:
Implement an expanding-window cross-validation strategy to robustly evaluate all existing models (SARIMA, XGBoost, ETS, Prophet, State Space).

### Walk-Forward Validation Setup

This section establishes the framework for **time-series cross-validation**.

Unlike random cross-validation, `TimeSeriesSplit` preserves chronological order. Each successive fold expands the training history and evaluates the model on a later unseen period.

A common MAPE calculation is also defined so that the different forecasting models can be evaluated consistently.

The goal is to obtain a more robust estimate of model performance than relying on a single train/test split.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Initialize TimeSeriesSplit with 5 folds for robust backtesting
tscv = TimeSeriesSplit(n_splits=5)

print(f'TimeSeriesSplit initialized with {tscv.n_splits} folds.')
# Verifying target for SARIMA/ETS and features for XGBoost
print(f'Target series length: {len(df)}')
print(f'Feature dataframe length: {len(feature_df_clean)}')

**comment**:
I will execute the walk-forward validation for SARIMA using the 5-fold TimeSeriesSplit. Each fold will refit the model on the expanding training set and forecast the next test window to calculate RMSE, MAE, and MAPE.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Target series
y_wfv = df.set_index('Date')['Close']
results_sarima = []

print('Starting SARIMA Walk-Forward Validation...')

for fold, (train_index, test_index) in enumerate(tscv.split(y_wfv)):
    y_train_cv, y_test_cv = y_wfv.iloc[train_index], y_wfv.iloc[test_index]

    # Fit SARIMA on expanding window
    model_cv = SARIMAX(y_train_cv, order=best_order, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    y_pred_cv = model_cv.forecast(steps=len(y_test_cv))

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test_cv, y_pred_cv))
    mae = mean_absolute_error(y_test_cv, y_pred_cv)
    mape = calculate_mape(y_test_cv, y_pred_cv)

    results_sarima.append({'Fold': fold+1, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape})
    print(f'Fold {fold+1}: RMSE={rmse:.2f}')

# Aggregation
sarima_wfv_df = pd.DataFrame(results_sarima)
avg_sarima_rmse = sarima_wfv_df['RMSE'].mean()
print(f'\nAverage SARIMA WFV RMSE: {avg_sarima_rmse:.2f}')

### XGBoost Walk-Forward Validation

This section applies the same expanding-window validation framework to XGBoost.

The causality-safe lag and rolling features are used as predictors, while the target remains the future closing price. For each validation fold, XGBoost is trained only on observations available before that fold and then evaluated on the subsequent unseen observations.

This tests whether the strong performance observed on the original hold-out set remains consistent across multiple historical periods.

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Prepare features and target from the cleaned audited dataframe
X_wfv = feature_df_clean[feature_cols]
y_wfv = feature_df_clean['Close']
results_xgb = []

print(f'Starting XGBoost Walk-Forward Validation ({tscv.n_splits} folds)...\n')

for fold, (train_index, test_index) in enumerate(tscv.split(feature_df_clean)):
    X_train_cv, X_test_cv = X_wfv.iloc[train_index], X_wfv.iloc[test_index]
    y_train_cv, y_test_cv = y_wfv.iloc[train_index], y_wfv.iloc[test_index]

    # Fit XGBoost
    xgb_cv = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    ).fit(X_train_cv, y_train_cv)

    xgb_pred = xgb_cv.predict(X_test_cv)

    rmse = np.sqrt(mean_squared_error(y_test_cv, xgb_pred))
    mae = mean_absolute_error(y_test_cv, xgb_pred)
    mape = calculate_mape(y_test_cv, xgb_pred)

    results_xgb.append({'Fold': fold+1, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape})
    print(f'Fold {fold+1}: RMSE={rmse:.2f}')

xgb_wfv_df = pd.DataFrame(results_xgb)
avg_xgb_rmse = xgb_wfv_df['RMSE'].mean()
print(f'\nAverage XGBoost WFV RMSE: {avg_xgb_rmse:.2f}')

### Additional Models in Walk-Forward Validation

This section extends the walk-forward evaluation to the remaining statistical and forecasting models: **ETS, Prophet, and State Space**.

Each model is evaluated on chronological validation windows using the same error metrics. The resulting fold-level scores are retained so that average performance can be compared fairly with SARIMA and the machine-learning models.

This creates a consistent multi-model validation framework across the complete forecasting pipeline.

**comment**:
I will fix the Prophet validation logic within the walk-forward loop by ensuring the 'ds' column is explicitly cast to datetime, satisfying Prophet's input requirements.

**comment**:
I am re-executing the walk-forward validation for ETS, State Space, and Prophet. I will use a robust loop that handles the date conversion for Prophet explicitly to avoid earlier errors and ensure all 5 folds are processed.

**comment**:
I will execute the complete walk-forward validation loop for ETS, State Space, and Prophet, ensuring the date formats are correctly handled for Prophet and that all 5 folds finish to provide a stable average RMSE for each model.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.structural import UnobservedComponents
from prophet import Prophet
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# Initialize results storage for the remaining models
results_others = {'ETS': [], 'Prophet_Default': [], 'Prophet_No_Season': [], 'State_Space': []}

print(f'Starting complete Walk-Forward Validation for remaining models ({tscv.n_splits} folds)...')

for fold, (train_index, test_index) in enumerate(tscv.split(y_wfv)):
    y_train_cv, y_test_cv = y_wfv.iloc[train_index], y_wfv.iloc[test_index]

    # 1. ETS - Additive Trend
    ets_cv = ExponentialSmoothing(y_train_cv, trend='add', seasonal=None, initialization_method='estimated').fit()
    ets_pred = ets_cv.forecast(len(y_test_cv))
    results_others['ETS'].append(np.sqrt(mean_squared_error(y_test_cv, ets_pred)))

    # 2. State Space - Local Linear Trend
    ss_cv = UnobservedComponents(y_train_cv, level='local linear trend').fit(disp=False)
    ss_pred = ss_cv.get_forecast(steps=len(y_test_cv)).predicted_mean
    results_others['State_Space'].append(np.sqrt(mean_squared_error(y_test_cv, ss_pred)))

    # 3. Prophet Processing
    p_df = y_train_cv.reset_index()
    p_df.columns = ['ds', 'y']
    p_df['ds'] = pd.to_datetime(p_df['ds'])
    p_future = pd.DataFrame({'ds': pd.to_datetime(y_test_cv.index)})

    # Default Prophet (Daily seasonality only as per initial analysis)
    m_def = Prophet(daily_seasonality=True, yearly_seasonality=False, weekly_seasonality=False)
    m_def.fit(p_df)
    forecast_def = m_def.predict(p_future)
    results_others['Prophet_Default'].append(np.sqrt(mean_squared_error(y_test_cv, forecast_def['yhat'])))

    # No-Seasonality Prophet
    m_no = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    m_no.fit(p_df)
    forecast_no = m_no.predict(p_future)
    results_others['Prophet_No_Season'].append(np.sqrt(mean_squared_error(y_test_cv, forecast_no['yhat'])))

    print(f'Fold {fold+1}/{tscv.n_splits} finished for all 4 models.')

# Calculate and Display Averages
print('\n--- Walk-Forward Validation Results Summary ---')
for model, rmse_list in results_others.items():
    print(f'{model:20} | Avg WFV RMSE: {np.mean(rmse_list):.2f}')

**comment**:
I will create a comprehensive summary table comparing the average walk-forward RMSE across all models (SARIMA, XGBoost, ETS, Prophet Default, Prophet No-Season, and State Space) and provide natural language insights to finish the subtask.

In [ ]:
import pandas as pd

# Consolidating results from the successful WFV runs
# SARIMA and XGBoost results were calculated in previous steps
# ETS, Prophet, and State Space were calculated in the most recent step

summary_data = [
    {'Model': 'XGBoost', 'Avg WFV RMSE': 49.88},
    {'Model': 'SARIMA', 'Avg WFV RMSE': 107.08},
    {'Model': 'State Space', 'Avg WFV RMSE': 153.26},
    {'Model': 'ETS', 'Avg WFV RMSE': 158.58},
    {'Model': 'Prophet (No Seasonality)', 'Avg WFV RMSE': 177.53},
    {'Model': 'Prophet (Default)', 'Avg WFV RMSE': 178.18}
]

leaderboard_df = pd.DataFrame(summary_data).sort_values('Avg WFV RMSE')

print('Phase 1: Walk-Forward Validation Performance Leaderboard')
print('='*60)
print(leaderboard_df.to_string(index=False))
print('='*60)

# Conclusion check against target
target_rmse = 15.77
best_rmse = leaderboard_df.iloc[0]['Avg WFV RMSE']

print(f'Best Performer: {leaderboard_df.iloc[0]["Model"]}')
print(f'Current Best WFV RMSE: {best_rmse:.2f}')
print(f'Target Baseline RMSE: {target_rmse}')

if best_rmse <= target_rmse:
    print('\\nSUCCESS: Model has met or exceeded the target benchmark.')
else:
    print('\\nNOTE: While XGBoost is the strongest, further tuning in Phase 3/4 is required to reach the target of 15.77.')

## SHAP Explainability

Use SHAP to interpret the champion gradient-boosting model and understand global and local feature effects.

**comment**:
I will begin by installing the SHAP library and calculating the SHAP values for the XGBoost model, which was identified as a top performer, to understand the impact of individual features on the forecast.

In [ ]:
import shap
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Using the champion XGBoost model from the previous validation
# Refit on the audited training data to ensure consistency
champion_xgb = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
).fit(X_train, y_train)

# Initialize Javascript for SHAP (necessary for some plots in notebooks)
shap.initjs()

# 1. Initialize the TreeExplainer
explainer = shap.TreeExplainer(champion_xgb)

# 2. Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# 3. Generate Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("SHAP Global Feature Importance - XGBoost Champion")
plt.tight_layout()
plt.show()

# 4. Generate Beeswarm Plot to show directionality
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Feature Impact (Directionality)")
plt.tight_layout()
plt.show()

**comment**:
I will create a local explanation using a SHAP Waterfall plot for the first sample in the test set. This visualization decomposes the prediction into the specific contribution of each feature relative to the base value (mean model prediction).

In [ ]:
import shap
import matplotlib.pyplot as plt

# To generate a Waterfall plot, we need an Explanation object
# We'll use the first sample of X_test for this local explanation

# 1. Create the Explanation object for the test set
explainer_local = shap.TreeExplainer(champion_xgb)
# shap_values for the test set were calculated previously, but we need the base value (expected value)
expected_value = explainer_local.expected_value

# 2. Re-calculate SHAP values specifically for the Waterfall plot object
# Using shap.Explainer is the standard way to get the .values and .base_values together
expl = shap.Explanation(values=shap_values,
                        base_values=expected_value,
                        data=X_test.values,
                        feature_names=X_test.columns.tolist())

# 3. Plot the Waterfall for the first instance in the test set
plt.figure(figsize=(10, 6))
shap.plots.waterfall(expl[0], show=False)
plt.title("Local Explanation (Waterfall Plot) - Test Instance #0")
plt.show()

print("Local explanation generated. This plot shows how specific feature values moved the prediction from the base value to the final output.")

## Section 7 — External Market Factors

Integrate NIFTY 50, Brent Crude Oil, and USD/INR as external market drivers. The repeated intermediate MultiIndex-fix attempts from the original notebook are removed; the final helper implementation is retained.

In [ ]:
import yfinance as yf
import pandas as pd

# Define the date range based on the existing df
start_date = df['Date'].min().strftime('%Y-%m-%d')
end_date = (df['Date'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

print(f"Fetching external data from {start_date} to {end_date}...")

# Download external drivers
nifty_df = yf.download('^NSEI', start=start_date, end=end_date)[['Close']].rename(columns={'Close': 'NIFTY50'})
crude_df = yf.download('CL=F', start=start_date, end=end_date)[['Close']].rename(columns={'Close': 'Crude_Oil'})
usdinr_df = yf.download('USDINR=X', start=start_date, end=end_date)[['Close']].rename(columns={'Close': 'USD_INR'})

print("\nNIFTY 50 Preview:")
print(nifty_df.head())
print("\nCrude Oil Preview:")
print(crude_df.head())
print("\nUSD/INR Preview:")
print(usdinr_df.head())


In [ ]:
import pandas as pd

# --- Helper Function: Process yfinance DataFrames ---
def fix_yfinance_df(df_ext, target_column_name, yf_ticker_symbol):
    # This function processes raw yfinance DataFrames, flattening MultiIndex columns
    # and extracting the 'Close' price for a given ticker, then renaming it.

    df_temp = df_ext.copy()

    # Step 1: Flatten MultiIndex columns if they exist. yfinance often returns columns
    # like ('Close', 'TICKER_SYMBOL') for single ticker requests.
    if isinstance(df_temp.columns, pd.MultiIndex):
        # Create new column names by joining the levels (e.g., 'Close_^NSEI')
        df_temp.columns = ['_'.join(col).strip() for col in df_temp.columns.values]

    # Step 2: Reset the index to convert the 'Date' index into a regular column.
    df_temp = df_temp.reset_index()

    # Step 3: Identify the column that contains the relevant 'Close' data.
    # This is the crucial part that caused the previous IndexError.
    # We'll search for the column that holds the actual close price data after yfinance's processing.

    # Based on the yf.download in cell 0befda95, the intent was to rename the 'Close'
    # column to `target_column_name` (e.g., 'NIFTY50'). However, sometimes yfinance
    # results in names like 'Close_TICKER_SYMBOL' or directly 'TICKER_SYMBOL' if it's the only one.
    # And from kernel state, 'NIFTY50_^NSEI' was observed.

    selected_close_col = None

    # Try to find the column by the target_column_name directly first
    if target_column_name in df_temp.columns:
        selected_close_col = target_column_name
    else:
        # Look for a column that matches the yfinance ticker symbol directly or includes 'Close'
        # e.g., 'NIFTY50_^NSEI' or 'CL=F'
        potential_cols = [
            col for col in df_temp.columns
            if yf_ticker_symbol.replace('^', '').replace('=', '') in col.replace('^', '').replace('=', '')
            and ('Close' in col or target_column_name in col)
        ]
        if potential_cols:
            selected_close_col = potential_cols[0]
        else:
            # Fallback for generic 'Close' if specific ticker match is not found
            close_cols = [c for c in df_temp.columns if 'Close' in c]
            if close_cols:
                selected_close_col = close_cols[0]
            else:
                # If still not found, and there's another column besides 'Date', assume it's the data column.
                if len(df_temp.columns) > 1 and df_temp.columns[1] != 'Date':
                    selected_close_col = df_temp.columns[1]

    if selected_close_col is None:
        raise ValueError(f"Could not find a suitable price column for {target_column_name} (ticker: {yf_ticker_symbol}) in yfinance data.")

    # Return a DataFrame containing only 'Date' and the identified 'Close' column,
    # with the 'Close' column renamed to the specified `target_column_name`.
    return df_temp[['Date', selected_close_col]].rename(columns={selected_close_col: target_column_name})

# --- Main Integration Logic: External Market Drivers ---

# Step 1: Process each external market data source using the helper function.
# This extracts and cleans the NIFTY 50, Crude Oil, and USD/INR 'Close' prices.
# Need to pass original yfinance ticker symbols for more robust identification in fix_yfinance_df.
nifty_final = fix_yfinance_df(nifty_df, 'NIFTY50', '^NSEI')
crude_final = fix_yfinance_df(crude_df, 'Crude_Oil', 'CL=F')
usdinr_final = fix_yfinance_df(usdinr_df, 'USD_INR', 'USDINR=X')

# Initialize a copy of the main Reliance stock dataframe (`df`) for integrating external data.
df_ext_integrated = df.copy()

# Define the list of new external columns that were merged.
ext_cols = ['NIFTY50', 'Crude_Oil', 'USD_INR']

# IMPORTANT: Remove existing external columns from df_ext_integrated before merging
# This prevents pandas from creating '_x' and '_y' suffixes if columns already exist,
# which would cause issues later when trying to ffill/bfill the original 'ext_cols'.
cols_to_drop = [col for col in ext_cols if col in df_ext_integrated.columns]
if cols_to_drop:
    df_ext_integrated = df_ext_integrated.drop(columns=cols_to_drop)

# Step 2: Merge the processed external dataframes sequentially with the primary dataframe.
for ext_df in [nifty_final, crude_final, usdinr_final]:
    # Ensure the 'Date' columns in both dataframes are consistent datetime types before merging.
    ext_df['Date'] = pd.to_datetime(ext_df['Date'])
    df_ext_integrated['Date'] = pd.to_datetime(df_ext_integrated['Date'])

    # Perform a left merge on the 'Date' column. This keeps all original Reliance dates
    # and adds the external market data where dates match.
    df_ext_integrated = df_ext_integrated.merge(ext_df, on='Date', how='left')

# Step 3: Handle any missing values introduced by market holiday mismatches.
# Use forward-fill (`ffill()`) to propagate the last valid observation forward,
# and then backward-fill (`bfill()`) to fill any remaining NaNs at the beginning of the series.
df_ext_integrated[ext_cols] = df_ext_integrated[ext_cols].ffill().bfill()

# --- Verification and Update Global DataFrame ---

# Print a confirmation message and display the sum of nulls for the new external columns
# to ensure successful filling.
print('External market drivers integrated successfully.')
print(df_ext_integrated[ext_cols].isnull().sum())

# Display the head of the updated dataframe to visually verify the integration of new columns.
display(df_ext_integrated[['Date', 'Close', 'NIFTY50', 'Crude_Oil', 'USD_INR']].head())

# Update the global `df` variable to reflect the dataframe with integrated external factors.
df = df_ext_integrated

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculate Correlation Matrix
corr_cols = ['Close', 'NIFTY50', 'Crude_Oil', 'USD_INR']
corr_matrix = df[corr_cols].corr()

# 2. Plot Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix: Reliance vs External Drivers')
plt.show()

# 3. Time Series Comparison Plot
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(df['Date'], df['Close'], color='blue')
axes[0].set_title('Reliance Close Price')

axes[1].plot(df['Date'], df['NIFTY50'], color='green')
axes[1].set_title('NIFTY 50')

axes[2].plot(df['Date'], df['Crude_Oil'], color='brown')
axes[2].set_title('Crude Oil Price')

axes[3].plot(df['Date'], df['USD_INR'], color='purple')
axes[3].set_title('USD/INR Exchange Rate')

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()

print('Correlation and Trend Analysis for external drivers completed.')

## Section 8 — Streamlit V2 Dashboard

Generate the Streamlit dashboard integrating market context, technical analysis, model performance, SHAP explainability, and forecasting outputs.

In [ ]:
with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import os

st.set_page_config(page_title="Reliance Industries - Pro Forecast", layout="wide")

OUTPUT_DIR = "/content/drive/My Drive/DS/Time_series_forecasting"

@st.cache_data
def load_data():
    path = os.path.join(OUTPUT_DIR, "Company_stock_prices_clean.csv")
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    return df.sort_values('Date')

@st.cache_resource
def load_artifacts():
    xgb = joblib.load(os.path.join(OUTPUT_DIR, "xgb_model.pkl"))
    cols = joblib.load(os.path.join(OUTPUT_DIR, "feature_cols.pkl"))
    return xgb, cols

df = load_data()
xgb_model, feature_cols = load_artifacts()

st.title("Reliance Industries Advanced Forecast Dashboard")
st.caption(f"Data as of: {df['Date'].max().date()}")

tabs = st.tabs(["Market Context", "Technical Analysis", "Model Performance", "SHAP Explainability", "30-Day Forecast"])

with tabs[0]:
    st.subheader("External Market Drivers")
    ext_cols = ['Close', 'NIFTY50', 'Crude_Oil', 'USD_INR']
    if all(c in df.columns for c in ext_cols):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.heatmap(df[ext_cols].corr(), annot=True, cmap='RdBu', ax=ax)
        st.pyplot(fig)
        st.info("Strong negative correlations observed between Reliance Close and Crude Oil/USD_INR.")

with tabs[1]:
    st.subheader("Technical Indicator Suite")
    tech_choice = st.selectbox("Select Indicator", ["Bollinger Bands", "RSI", "MACD"])
    fig, ax = plt.subplots(figsize=(12, 5))
    if tech_choice == "Bollinger Bands":
        ax.plot(df['Date'], df['Close'], label='Close', alpha=0.6)
        ax.plot(df['Date'], df['BB_Upper'], 'r--', label='Upper Band')
        ax.plot(df['Date'], df['BB_Lower'], 'g--', label='Lower Band')
        ax.fill_between(df['Date'], df['BB_Lower'], df['BB_Upper'], alpha=0.1)
    elif tech_choice == "RSI":
        ax.plot(df['Date'], df['RSI'], color='purple')
        ax.axhline(70, color='red', linestyle='--')
        ax.axhline(30, color='green', linestyle='--')
    elif tech_choice == "MACD":
        ax.bar(df['Date'], df['MACD'], label='MACD')
        ax.plot(df['Date'], df['MACD_Signal'], color='orange', label='Signal')
    ax.legend()
    st.pyplot(fig)

with tabs[2]:
    st.subheader("Walk-Forward Validation Metrics")
    metrics_path = os.path.join(OUTPUT_DIR, "model_comparison.csv")
    if os.path.exists(metrics_path):
        st.table(pd.read_csv(metrics_path, index_col=0))

with tabs[3]:
    st.subheader("Model Decision Interpretability (SHAP)")
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(df[feature_cols].tail(100))
    fig, ax = plt.subplots()
    shap.summary_plot(shap_values, df[feature_cols].tail(100), show=False)
    st.pyplot(fig)

with tabs[4]:
    st.subheader("30-Day Recursive Forecast")
    if st.button("Generate Forecast"):
        history = df.tail(20).copy()
        forecasts = []
        for _ in range(30):
            row = history[feature_cols].tail(1)
            pred = xgb_model.predict(row)[0]
            forecasts.append(pred)
            new_row = history.tail(1).copy()
            new_row['Close'] = pred
            history = pd.concat([history, new_row])
        st.line_chart(forecasts)
        st.success("30-Day Forecast Generated via Champion XGBoost.")
''')

print('Streamlit V2 Dashboard Expansion completed in app.py.')

## Section 9 — MLflow Experiment Tracking & Production Export

Track the trained gradient-boosting models with MLflow and organize datasets, models, and metrics into a production-ready directory structure.

In [ ]:
import os
import joblib
import mlflow
import mlflow.xgboost
import mlflow.sklearn # Import mlflow.sklearn for logging general sklearn-compatible models
import mlflow.lightgbm # Import mlflow.lightgbm for specific LightGBM logging
import mlflow.catboost # Import mlflow.catboost for specific CatBoost logging
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
import shap # Added for SHAP integration

# --- BEGIN PATCH FOR NameError: name 'df'/'df_tech_final' is not defined ---
# This block is added to handle cases where previous cells defining 'df' or 'df_tech_final'
# were not executed. For full functionality, ensure the entire notebook is run top-to-bottom.
try:
    _ = df  # Attempt to access df
except NameError:
    print("WARNING: DataFrame 'df' not found. Creating an empty placeholder. Please run all preceding cells for full functionality.")
    df = pd.DataFrame({'Date': pd.to_datetime([]), 'Open': [], 'High': [], 'Low': [], 'Close': [], 'Adj Close': [], 'Volume': []}) # Minimal structure for export script

try:
    _ = df_tech_final # Attempt to access df_tech_final
except NameError:
    print("WARNING: DataFrame 'df_tech_final' not found. Creating an empty placeholder. Please run all preceding cells for full functionality.")
    df_tech_final = pd.DataFrame({'Date': pd.to_datetime([])}) # Minimal structure for merge later
# --- END PATCH ---

# Define constants for file paths
BASE_DIR = "." # Running from repo root
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "models")
METRICS_DIR = os.path.join(BASE_DIR, "metrics")

# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# --- 1. Merge all features into the final dataframe that gets deployed ---
# Ensure df (Reliance main data) and df_tech_final (technical indicators) are available
# Also assumes external market factors (NIFTY50, Crude_Oil, USD_INR) are already in df
final_df = df.copy()

# Add technical indicators from df_tech_final, if not already present
# Make sure 'Date' is datetime in both for proper merge
if 'Date' in final_df.columns:
    final_df['Date'] = pd.to_datetime(final_df['Date'])
if 'Date' in df_tech_final.columns:
    df_tech_final['Date'] = pd.to_datetime(df_tech_final['Date'])

# Define common columns to exclude from merging twice or as features
# These are the original OHLCV and calculated daily returns/MAs
base_cols_excluding_features = [
    'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
    'Daily_Return', 'MA_20', 'MA_200', 'Volatility_20', 'Extreme_Movement', 'Close_Ratio'
]

# Identify columns in df_tech_final that are not in final_df (excluding base_cols)
tech_cols_to_add = [
    col for col in df_tech_final.columns
    if col not in final_df.columns and col not in base_cols_excluding_features
]

if tech_cols_to_add:
    # Merge only the new technical columns + Date
    final_df = final_df.merge(
        df_tech_final[['Date'] + tech_cols_to_add],
        on='Date',
        how='left'
    )
    # Forward-fill and back-fill any NaNs that might arise from merging due to initial NaN rows in df_tech_final
    for col in tech_cols_to_add:
        final_df[col] = final_df[col].ffill().bfill()

# Ensure all feature_cols are available in final_df
# The `feature_cols` list is defined for XGBoost and other tree models
# It might contain lagged features etc. which were generated on `df_original_clean` or `feature_df_clean`
# For a robust final_df, re-generate these on `final_df` if they aren't directly carried over from merges.
# This part is implicit and assumes `feature_df_clean` (from Phase 2 Leakage Audit)
# has been properly integrated into `final_df` through the `df = df_ext_integrated` step in Phase 7.
# Let's verify `feature_cols` from the environment.

# Re-create the feature_cols list from the current global state (assuming it's loaded from a previous cell)
# The `feature_cols` variable should be the list used for XGBoost and other models.
# Re-confirming it is available from previous runs.

# Saving the cleaned and feature-engineered dataframe
final_df.to_csv(os.path.join(DATA_DIR, "Company_stock_prices_clean.csv"), index=False)

# --- 2. Save every trained model artifact ---
# Use globals().get() to safely access model variables, as some might not be run
artifacts = {
    "xgb_model.pkl": globals().get('xgb_final', None) or globals().get('xgb_model', None), # Assuming xgb_final from cell 81 or xgb_model from earlier
    "sarima_model.pkl": globals().get('sarima_final', None) or globals().get('sarima_model', None),
    "ets_model.pkl": globals().get('ets_final', None) or globals().get('ets_model', None),
    "state_space_model.pkl": globals().get('state_space_final', None) or globals().get('state_space_model', None),
    "lgbm_model.pkl": globals().get('lgbm_model', None),
    "cat_model.pkl": globals().get('cat_model', None),
    "feature_cols.pkl": globals().get('feature_cols', None) # feature_cols needs to be saved
}

# Prophet models have their own saving needs
prophet_default_model = globals().get('prophet_default_final', None) or globals().get('prophet_default', None)
prophet_no_season_model = globals().get('prophet_no_seasonality_final', None) or globals().get('prophet_no_seasonality', None)

if prophet_default_model:
    joblib.dump(prophet_default_model, os.path.join(MODELS_DIR, "prophet_default_model.pkl"))
if prophet_no_season_model:
    joblib.dump(prophet_no_season_model, os.path.join(MODELS_DIR, "prophet_no_seasonality_model.pkl"))

for filename, obj in artifacts.items():
    if obj is not None:
        joblib.dump(obj, os.path.join(MODELS_DIR, filename))
    else:
        print(f"⚠️  Skipped {filename} — variable not found in kernel. Please ensure its training cell was run.")

# Save the comparison_df. This assumes `comparison_df` is globally available.
_comparison_df_obj = globals().get('comparison_df', None)
if _comparison_df_obj is not None:
    _comparison_df_obj.to_csv(os.path.join(METRICS_DIR, "model_comparison.csv"))
else:
    print("⚠️  Skipped saving model_comparison.csv — `comparison_df` not found in kernel.")

# --- SHAP Artifacts for XGBoost Champion Model ---
# Assuming `champion_xgb` is the final trained XGBoost model from Phase 9 or earlier.
# And `X_test` and `feature_cols` are globally available for SHAP.
champion_xgb_shap = globals().get('champion_xgb', None) or globals().get('xgb_model', None) # Try to get the champion XGBoost model
X_test_shap = globals().get('X_test', None)
feature_cols_shap = globals().get('feature_cols', None)

if champion_xgb_shap is not None and X_test_shap is not None and feature_cols_shap is not None:
    # Sample X_test for SHAP to avoid large file sizes/slow loading in Streamlit
    sample_X_test_for_shap = X_test_shap[feature_cols_shap].sample(
        min(200, len(X_test_shap)), random_state=42
    )

    explainer = shap.TreeExplainer(champion_xgb_shap)
    shap_values_xgb = explainer.shap_values(sample_X_test_for_shap)

    joblib.dump(explainer, os.path.join(MODELS_DIR, "xgb_shap_explainer.pkl"))
    joblib.dump(shap_values_xgb, os.path.join(MODELS_DIR, "xgb_shap_values.pkl"))
    joblib.dump(sample_X_test_for_shap, os.path.join(MODELS_DIR, "xgb_shap_X_test_sample.pkl"))
else:
    print("⚠️  Skipped SHAP artifact saving — `champion_xgb`, `X_test`, or `feature_cols` not found.")

# --- 3. MLflow: log all three gradient-boosting models (XGBoost, LightGBM, CatBoost) ---
mlflow.set_experiment("Reliance_Stock_Forecasting")

# Define trusted types for skops.io serialization (used by mlflow.sklearn.log_model for LightGBM/CatBoost)
# Ensure all models are available for logging
xgb_mlflow_model = globals().get('xgb_final', None) or globals().get('xgb_model', None) # Get final trained models
lgbm_mlflow_model = globals().get('lgbm_model', None)
cat_mlflow_model = globals().get('cat_model', None)

gb_models_to_log = {
    "XGBoost": xgb_mlflow_model,
    "LightGBM": lgbm_mlflow_model,
    "CatBoost": cat_mlflow_model
}

# The X_test and y_test from the latest split (after full feature engineering)s
X_test_mlflow = globals().get('X_test', None)
y_test_mlflow = globals().get('y_test', None)


if X_test_mlflow is None or y_test_mlflow is None:
    print("⚠️  Skipped MLflow logging for GB models — X_test or y_test not found.")
else:
    skops_trusted_types = [
        'collections.OrderedDict', # For general Python objects
        'lightgbm.basic.Booster', # LightGBM's internal model object
        'lightgbm.sklearn.LGBMRegressor', # LightGBM's scikit-learn wrapper
        'catboost.core.CatBoostRegressor', # CatBoost's model object
        'xgboost.sklearn.XGBRegressor' # XGBoost's scikit-learn wrapper
    ]

    for name, model in gb_models_to_log.items():
        if model is None:
            print(f"Skipping MLflow logging for {name} - model object not found in global scope.")
            continue
        try:
            with mlflow.start_run(run_name=f"{name}_Final"):
                preds = model.predict(X_test_mlflow)
                rmse = np.sqrt(mean_squared_error(y_test_mlflow, preds))
                mae = mean_absolute_error(y_test_mlflow, preds)
                mape = np.mean(np.abs((y_test_mlflow - preds) / y_test_mlflow)) * 100

                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("mae", mae)
                mlflow.log_metric("mape", mape)

                log_args = {
                    "artifact_path": "model",
                    "registered_model_name": f"{name}_Reliance_Forecast",
                    "signature": mlflow.models.signature.infer_signature(X_test_mlflow, preds),
                    "input_example": X_test_mlflow.head(2),
                    "metadata": {"skops_trusted_types": skops_trusted_types} # Keep this for now, may be used by generic loggers
                }

                if name == "XGBoost":
                    mlflow.xgboost.log_model(model, **log_args)
                elif name == "LightGBM":
                    # Use specialized LightGBM logger
                    mlflow.lightgbm.log_model(model, **log_args)
                elif name == "CatBoost":
                    # Use specialized CatBoost logger
                    mlflow.catboost.log_model(model, **log_args)
                else:
                    # Fallback for other sklearn-compatible models if any were added
                    mlflow.sklearn.log_model(model, **log_args)

                print(f"MLflow: Logged {name} — RMSE {rmse:.2f}")
        except Exception as e:
            print(f"ERROR during MLflow logging for {name}: {e}")

# Final confirmation message with directory structure
print(f"\n✅ Export complete.")
print(f"   Data:    {DATA_DIR}/")
print(f"   Models:  {MODELS_DIR}/")
print(f"   Metrics: {METRICS_DIR}/")
print(f"   All files (models, data, metrics) are now organized under {BASE_DIR}/")